In [1]:
from groq import Groq
from dotenv import load_dotenv
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [2]:
import json

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex


reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [3]:
def search(query):
    results = index.search(query=query, num_results=5)
    return results

def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

In [4]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search the documentation database for relevant results based on a query string.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query to look up in the index"
                    }
                },
                "required": ["query"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function", 
        "function": {
            "name": "add_entry",
            "description": "Add a new documentation entry to the index.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {"type": "string", "description": "The source filename"},
                    "title": {"type": "string", "description": "The title of the entry"},
                    "description": {"type": "string", "description": "A short description"},
                    "content": {"type": "string", "description": "The full content"}
                },
                "required": ["filename", "title", "description", "content"],
                "additionalProperties": False
            }
        }
    }
]


In [ ]:
def make_call(tool_call):
    """Execute the function based on tool call"""
    try:
        arguments = json.loads(tool_call.function.arguments)
        name = tool_call.function.name
        
        if name == 'search':
            result = index.search(**arguments) 
        elif name == 'add_entry':
            result = add_entry(**arguments)
        else: 
            result = f'Unknown tool: "{name}"'

        # Truncate to avoid 413
        result_str = json.dumps(result)
        if len(result_str) > 2000:
            result_str = result_str[:2000] + '... [truncated]'    
        
        return {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": str(result_str),
        }
    except Exception as e:
        print(f"Error in make_call: {e}")
        return {
            "role": "tool", 
            "tool_call_id": tool_call.id,
            "content": f"Error: {str(e)}",
        }

In [14]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [7]:
test_messages = [
    {"role": "system", "content": "You are a helpful assistant with access to search and add_entry tools."},
    {"role": "user", "content": "How do I create a dashboard?"}
]

print("Testing API call...")
try:
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",  # Try the 8B model instead
        messages=test_messages,
        tools=tools,
        tool_choice="auto",
        temperature=0.7,
        max_tokens=100,  # Limit for testing
    )
    print("SUCCESS!")
    print(response.choices[0].message)
except Exception as e:
    print(f"FAILED: {e}")

Testing API call...
SUCCESS!
ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=[ChatCompletionMessageToolCall(id='7gzgqsaby', function=Function(arguments='{"query":"create dashboard"}', name='search'), type='function')])


In [8]:
# Initialize conversation
message_history = [
    {"role": "system", "content": instructions},
]

question = "How do I create a dashboard in Evidently?"
message_history.append({"role": "user", "content": question})

max_iterations = 4
iteration_count = 0

while iteration_count < max_iterations:
    print(f"\n--- Iteration {iteration_count + 1} ---")
    
    # Call Groq with tools
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",  # Using the working model
        messages=message_history,
        tools=tools,
        tool_choice="auto",
        temperature=0.7,
    )
    
    assistant_message = response.choices[0].message
    
    def truncate_message(content, max_length=2000):
        """Truncate long messages to prevent context overflow"""
        if content and len(content) > max_length:
            return content[:max_length] + "... [truncated]"
        return content

    # Create clean message for history (without annotations)
    clean_message = {
        "role": assistant_message.role,
        "content": truncate_message(assistant_message.content, max_length=1500),
    }
    if assistant_message.tool_calls:
        clean_message["tool_calls"] = assistant_message.tool_calls
    
    message_history.append(clean_message)
    
    # Check for tool calls
    if assistant_message.tool_calls:
            # Before executing, check if the assistant also provided text
        print(f"💭 Explanation: {assistant_message.content}")
        print(f"🤖 Model requested: {len(assistant_message.tool_calls)} tool call(s)")
        
        # Execute each tool call
        for tool_call in assistant_message.tool_calls:
            print(f"📞 Executing {tool_call.function.name}...")
            tool_result = make_call(tool_call)
            message_history.append(tool_result)
            print(f"✅ Completed {tool_call.function.name}")
        
    elif assistant_message.content:
        print("\n" + "="*50)
        print("ASSISTANT FINAL ANSWER:")
        print("="*50)
        print(assistant_message.content)
        break
    
    iteration_count += 1

if iteration_count >= max_iterations:
    print("Reached max iterations without final answer")


--- Iteration 1 ---
💭 Explanation: To create a dashboard in Evidently, you'll need to follow these general steps: 
1. Install the Evidently library and import it in your project.
2. Create a data source to connect to your model's data.
3. Define metrics and visualizations for your dashboard.
4. Use the library's APIs to build and customize your dashboard.


🤖 Model requested: 1 tool call(s)
📞 Executing search...
✅ Completed search

--- Iteration 2 ---

ASSISTANT FINAL ANSWER:
To create a dashboard in Evidently, you can follow these steps:

1. First, add Reports with evaluation results to the Project.
2. Enter "Edit" mode on the Dashboard (top right corner).
3. Click the plus sign with "add Panel" on the left.
4. Choose a Panel type, such as text, counter, or distribution.
5. Select the Metrics or Presets that you want to display in the Panel.
6. Customize the Panel as needed, such as adding a title or changing the layout.
7. To add multiple panels, repeat steps 3-6.
8. To add tabs, cl

In [9]:
# Test with versatile model (fails)
try:
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": "Search for dashboard"}],
        tools=tools,
    )
    print("Versatile works?")
except Exception as e:
    print(f"Versatile fails: {e}")

# Test with instant model (works)
response = groq_client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Search for dashboard"}],
    tools=tools,
)
print("Instant works!")

Versatile fails: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '<function=search {"query": "dashboard"} </function>'}}
Instant works!


In [10]:
print("📚 Evidently Documentation Assistant")
print("Type 'stop' to exit, 'reset' to clear conversation")
print("-" * 50)

message_history = [
    {"role": "system", "content": instructions},
]

while True:
    user_input = input("\nYou: ").strip()
    
    if user_input.lower() == 'stop':
        print("Goodbye!")
        break
    
    if user_input.lower() == 'reset':
        message_history = [{"role": "system", "content": instructions}]
        print("Conversation reset!")
        continue
    
    message_history.append({"role": "user", "content": user_input})
    
    iteration_count = 0
    max_iterations = 5
    
    while iteration_count < max_iterations:
        print(f"\n[Step {iteration_count + 1}]")
        
        response = groq_client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=message_history,
            tools=tools,
            tool_choice="auto",
            temperature=0.7,
        )
        
        assistant_message = response.choices[0].message
        
        # Clean and append
        clean_message = {"role": assistant_message.role, "content": assistant_message.content}
        if assistant_message.tool_calls:
            clean_message["tool_calls"] = assistant_message.tool_calls
        
        message_history.append(clean_message)
        
        if assistant_message.tool_calls:
            print(f"🔧 Executing {len(assistant_message.tool_calls)} tool(s)...")
            for tool_call in assistant_message.tool_calls:
                tool_result = make_call(tool_call)
                message_history.append(tool_result)
                print(f"   ✓ {tool_call.function.name}")
        else:
            print(f"\n🤖 Assistant: {assistant_message.content}")
            break
        
        iteration_count += 1

📚 Evidently Documentation Assistant
Type 'stop' to exit, 'reset' to clear conversation
--------------------------------------------------

[Step 1]
🔧 Executing 1 tool(s)...
   ✓ search

[Step 2]

🤖 Assistant: The term "evidently" refers to a clear or obvious fact, or a statement that is easy to see or understand. It is often used to indicate that something is apparent or self-evident. In the context of the provided search results, "evidently" is used to describe a breaking change notice for a cloud platform update.

[Step 1]
🔧 Executing 1 tool(s)...
   ✓ search

[Step 2]

🤖 Assistant: The evidently package is a Python library that provides a platform for model monitoring and evaluation. It allows users to track and analyze the performance of their machine learning models, including metrics such as accuracy, precision, recall, and F1 score. The package also includes features for data drift detection and model interpretability. The Evidently Cloud platform provides a web-based interface 

In [25]:
class Agent:

    def __init__(self, llm_client, model, instructions, tools):
        self.llm_client = llm_client
        self.model = model
        self.instructions = instructions
        self.tools = tools

    def make_call(self,tool_call):
        """Execute the function based on tool call"""
        try:
            arguments = json.loads(tool_call.function.arguments)
            name = tool_call.function.name
            
            if name == 'search':
                result = search(**arguments) 
            elif name == 'add_entry':
                result = add_entry(**arguments)
            else: 
                result = f'Unknown tool: "{name}"'

            # Truncate to avoid 413
            result_str = json.dumps(result)
            if len(result_str) > 2000:
                result_str = result_str[:2000] + '... [truncated]'    
            
            return {
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(result_str),
            }
        except Exception as e:
            print(f"Error in make_call: {e}")
            return {
                "role": "tool", 
                "tool_call_id": tool_call.id,
                "content": f"Error: {str(e)}",
            }   

    def loop(self, user_prompt, message_history=None):
        if not message_history:
            message_history = [
                {"role": "system", "content": self.instructions},
            ]
            
        message_history.append({"role": "user", "content": user_prompt})

        iteration_number = 0
    
        while True:
            response = self.llm_client.chat.completions.create(
                model=self.model,
                messages=message_history,
                tools=self.tools,
                tool_choice="auto",
                parallel_tool_calls=False,
            )
        
            assistant_message = response.choices[0].message

            clean_message = {"role": assistant_message.role, "content": assistant_message.content}
            if assistant_message.tool_calls:
                clean_message["tool_calls"] = assistant_message.tool_calls
            message_history.append(clean_message)

            has_function_calls = False

            for tool_call in (assistant_message.tool_calls or []):
                print(f'executing {tool_call.function.name}({tool_call.function.arguments})...')
                tool_call_output = self.make_call(tool_call)
                message_history.append(tool_call_output)
                has_function_calls = True

            if not has_function_calls:
                if assistant_message.content:
                    print('ASSISTANT:', assistant_message.content)
                break
                    
            iteration_number = iteration_number + 1
            print()
            
            if not has_function_calls:
                break

        return message_history        

    def qna(self):
        message_history = [
            {"role": "system", "content": self.instructions},
        ]
        
        iteration_number = 1

        # Q&A loop
        while True:
            user_prompt = input('You:')
            if user_prompt.lower().strip() == 'stop':
                break
            
            # print(f"History length before loop: {len(message_history)}")
            message_history = self.loop(user_prompt, message_history)
            # print(f"History length after loop: {len(message_history)}")

In [26]:
agent = Agent(
    llm_client=groq_client,
    model="llama-3.1-8b-instant",
    # model="llama-3.3-70b-versatile",
    instructions=instructions,
    tools=tools
)

agent.qna()

executing add_entry({"content":"To learn Evidently, one approach is to begin with the official documentation. The documentation provides a comprehensive guide to getting started with Evidently, including tutorials and examples on how to implement it in your data science projects. By following these resources, you can develop a solid understanding of the fundamentals of Evidently and how it can be applied to real-world problems.\n\nAnother way to learn Evidently is by taking online courses or attending workshops. These educational resources offer hands-on experience and instruction from expert instructors, providing a structured learning path for learners of all levels. Furthermore, online communities and forums dedicated to Evidently can also serve as valuable learning resources, offering support, guidance, and connections with other practitioners.\n\nFinally, practice and hands-on experience with Evidently are essential for mastering the tool. By working on projects that involve data 